# Introduction to MONAI

MONAI is a deep learning library for medical imaging research. It provides a comprehensive set of tools and utilities for building, training, and evaluating deep learning models in the medical imaging domain.

## Key Features

1. **Data Loading and Preprocessing**: MONAI offers a wide range of data loading and preprocessing functionalities, including support for various image formats, normalization, and augmentation techniques.
2. **Transformations**: It includes a rich set of transformations for data augmentation, normalization, and other preprocessing steps.
3. **Model Building**: MONAI provides a modular approach to building deep learning models, including popular architectures like U-Net, V-Net, and Residual Networks.
4. **Training and Evaluation**: It offers tools for training models, including loss functions, optimizers, and evaluation metrics, making it easier to develop and deploy medical image segmentation models.
5. **Integration with PyTorch**: MONAI is built on top of PyTorch, allowing for seamless integration with the PyTorch ecosystem and leveraging its powerful ecosystem of tools and libraries.

## Installation

MONAI is already installed in the DL_labs_GPU environment on MAIA, and it is included in `requirements.txt` for your own computer. Some features need extra packages; the ones we use here (`nibabel` to read NIfTI files and `scipy` for the distance metrics) are in `requirements.txt` too. We will do the same as we did in PyTorch in the notebook "Segmentation and UNET". The idea is you will see the difference between the two frameworks and how MONAI may simplify the process of building and training deep learning models for medical imaging tasks.

In the second part of this notebook, you will train a 3D U-Net on the whole volumes, the way segmentation models are usually trained in practice.

In [ ]:
# Some imports

import monai
import torch
import matplotlib.pyplot as plt

from cm2003.decathlon_dataset import get_decathlon_dataloader
from cm2003.train import train

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Apple-silicon Mac
else:
    device = torch.device('cpu')
print('using device:', device)

In [ ]:
root_dir = './cm2003/datasets'
task = "Task04_Hippocampus"

# The first time, this downloads the dataset (about 30 MB) into cm2003/datasets/
train_loader = get_decathlon_dataloader(root_dir, task, "training", batch_size=4, num_workers=0, shuffle=True)
val_loader = get_decathlon_dataloader(root_dir, task, "validation", batch_size=4, num_workers=0, shuffle=False)

In [ ]:
# Load UNET model from MONAI

model = monai.networks.nets.UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
)

# Define loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Implement the training loop for the model with the same structure as for PyTorch. You may even make use of the train function we wrote in PyTorch in `cm2003/train.py`.

In [ ]:
from cm2003.train import train
# Your code here
trained_model = None



Now let's evaluate the model on the validation set with the metrics in MONAI. We give the example with Hausdorff Distance but you should try to find other metrics in the documentation: [https://docs.monai.io/en/stable/metrics.html](https://docs.monai.io/en/stable/metrics.html) appropriate for the segmentation task. We can interpret the Hausdorff Distance as the maximum difference between the predicted segmentation boundary and the true boundary for 95% of the points. In this task, we are trying to segment the hippocampus, which is a small structure. The distance is measured in pixels of the resized images, which here are roughly 1 mm in size. If you get `inf`, the model has missed a structure completely in at least one slice where it is present (the distance to an empty prediction is infinite); this happens with models that have not been trained enough.

In [ ]:
import warnings
import torch.nn.functional as F
from monai.metrics import HausdorffDistanceMetric

hausdorff_distance = HausdorffDistanceMetric(include_background=False, reduction="mean", percentile=95)

trained_model.eval()
with torch.no_grad(), warnings.catch_warnings():
    # Many slices do not contain the hippocampus at all. For those, the distance is undefined (nan)
    # and MONAI warns about it; reduction="mean" ignores the nan values when aggregating.
    warnings.filterwarnings("ignore", message=".*is all 0.*")
    # MONAI 1.5 warns about a deprecated argument it uses internally
    warnings.filterwarnings("ignore", message=".*always_return_as_numpy.*")
    for val_data in val_loader:
        val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
        # One-hot encode the predictions and labels: (B, 3, H, W)
        val_outputs = F.one_hot(trained_model(val_inputs).argmax(dim=1), num_classes=3).permute(0, 3, 1, 2)
        val_labels = F.one_hot(val_labels.long().squeeze(1), num_classes=3).permute(0, 3, 1, 2)
        hausdorff_distance(val_outputs, val_labels)

result = hausdorff_distance.aggregate().item()
hausdorff_distance.reset()
print(f'95th percentile Hausdorff distance: {result:.4f}')

## Evaluation for two other metrics

**Implement evaluation for two other metrics in MONAI and comment on the results.**

Now we have trained and evaluated the model, let's change the loss function to DiceLoss and train the model again which is another loss function commonly used in medical image segmentation. Look at the documentation of MONAI to find out how to do this: [https://docs.monai.io/en/stable/losses.html](https://docs.monai.io/en/stable/losses.html)

Hint: our labels are class indices of shape (B, 1, H, W) and the model outputs logits, so use `to_onehot_y=True` and `softmax=True`. The `train` function in `cm2003/train.py` works with MONAI losses as well. You can also try `DiceCELoss`, which combines Dice and cross-entropy and is a common default today.

After you have trained the model, you can evaluate the model on the validation set with the new loss function.

In [ ]:
# Your code here

# Part 2: 3D segmentation with MONAI

So far we have treated the hippocampus volumes as a stack of independent 2D slices, which were also resized to 32x64x32 voxels. This is simple, but it has two drawbacks:

- A 2D model never sees the neighbouring slices, although the hippocampus is a 3D structure and the slices above and below carry a lot of information.
- Resizing every volume to the same shape distorts the anatomy (the volumes have different sizes) and means our distances are no longer in millimetres.

In practice, medical image segmentation is usually done in 3D, as in nnU-Net and the MONAI tutorials:

1. **Native resolution:** the volumes are resampled to a fixed voxel spacing (here 1 mm), but not resized, so they keep their own shape (between 24 and 59 voxels along each axis).
2. **Patch-based training:** the model is trained on random 3D patches (here 32x32x32). This keeps the memory use under control for large scans, works with volumes of different sizes and acts as a form of data augmentation. `RandCropByPosNegLabeld` samples half of the patches around the hippocampus, so the model sees enough of the small foreground.
3. **Data augmentation** with MONAI's random 3D transforms.
4. **Sliding window inference:** at test time, the model is applied to overlapping patches that cover the whole volume and the predictions are stitched together (`monai.inferers.sliding_window_inference`).
5. **Evaluation per volume and per class:** the metrics are computed for each patient's whole 3D segmentation and each structure separately, which is what is reported in papers. Since the voxels are 1 mm, the Hausdorff distance is now in millimetres.

The data pipeline is in `get_3d_transforms` in `cm2003/decathlon_dataset.py`; have a look at it before you continue.

In [ ]:
from monai.transforms import RandFlipd, RandAffined, RandScaleIntensityd, RandShiftIntensityd
from cm2003.decathlon_dataset import get_decathlon_3d_dataloader, PATCH_SIZE

# TODO: choose data augmentations for the training patches. See the "Spatial" and "Intensity"
# dictionary transforms (the ones ending in "d") in https://docs.monai.io/en/stable/transforms.html
# Spatial transforms must be applied to both keys=["image", "label"] (use mode=("bilinear", "nearest")
# for transforms that interpolate, so the labels stay integers); intensity transforms only to keys="image".
augmentations = [
    # e.g. RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
]

train_loader_3d = get_decathlon_3d_dataloader(root_dir, task, "training", batch_size=2, augmentations=augmentations)
val_loader_3d = get_decathlon_3d_dataloader(root_dir, task, "validation", batch_size=1)

batch = next(iter(train_loader_3d))
print("Training batch:", batch["image"].shape)  # (number of patches, channels, x, y, z)
print("Validation volume:", next(iter(val_loader_3d))["image"].shape)

The 3D U-Net in MONAI is the same class as before, with `spatial_dims=3`. We train it with `DiceCELoss`, the sum of the Dice and cross-entropy losses, which is the default choice in nnU-Net and many MONAI tutorials. The functions below validate the model on whole volumes with sliding window inference, and train it while keeping the weights with the best validation Dice.

In [ ]:
import copy
import time
import warnings
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric, HausdorffDistanceMetric

CLASS_NAMES = ["anterior", "posterior"]


def evaluate_3d(model, loader):
    """Segment every validation volume with sliding window inference and compute the Dice score and
    95th percentile Hausdorff distance (in mm) per volume and per class (background excluded)."""
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
    hd_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")
    model.eval()
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*always_return_as_numpy.*")
        for data in loader:
            image, label = data["image"].to(device), data["label"].to(device)
            logits = sliding_window_inference(image, roi_size=PATCH_SIZE, sw_batch_size=8, predictor=model, overlap=0.5)
            pred = F.one_hot(logits.argmax(dim=1), num_classes=3).permute(0, 4, 1, 2, 3)
            target = F.one_hot(label.long().squeeze(1), num_classes=3).permute(0, 4, 1, 2, 3)
            dice_metric(pred, target)
            hd_metric(pred, target)
    dice, hd = dice_metric.aggregate().cpu(), hd_metric.aggregate().cpu()
    return {"dice": dice.tolist(), "hd95": hd.tolist(), "mean_dice": dice.mean().item()}


def train_3d(model, loss_fn, optimizer, epochs, val_every=5, scheduler=None):
    """Train a 3D model on patches and validate it on whole volumes every `val_every` epochs.
    Returns the model with the weights of the best validation epoch, and the training history."""
    history = {"train_loss": [], "val_epoch": [], "val_dice": []}
    best_dice, best_state = -1.0, None
    model.to(device)
    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()
        total_loss = 0.0
        for batch in train_loader_3d:
            images, labels = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if scheduler is not None:
            scheduler.step()
        history["train_loss"].append(total_loss / len(train_loader_3d))
        message = f"Epoch {epoch}/{epochs}: train loss {history['train_loss'][-1]:.4f}"

        if epoch % val_every == 0 or epoch == epochs:
            result = evaluate_3d(model, val_loader_3d)
            history["val_epoch"].append(epoch)
            history["val_dice"].append(result["mean_dice"])
            message += f", val Dice {result['mean_dice']:.4f}"
            if result["mean_dice"] > best_dice:
                best_dice, best_state = result["mean_dice"], copy.deepcopy(model.state_dict())
                message += " (best)"
        print(message + f" [{time.time() - start:.0f}s]")

    model.load_state_dict(best_state)
    return model, history

In [ ]:
model_3d = monai.networks.nets.UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
)
loss_fn_3d = monai.losses.DiceCELoss(to_onehot_y=True, softmax=True)
optimizer_3d = torch.optim.AdamW(model_3d.parameters(), lr=1e-3, weight_decay=1e-4)
epochs_3d = 40
scheduler_3d = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_3d, T_max=epochs_3d)

model_3d, history_3d = train_3d(model_3d, loss_fn_3d, optimizer_3d, epochs_3d, val_every=5, scheduler=scheduler_3d)
torch.save(model_3d.state_dict(), "best_unet3d.pth")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(history_3d["train_loss"]) + 1), history_3d["train_loss"])
plt.xlabel("Epoch"); plt.ylabel("Training loss (DiceCE)")
plt.subplot(1, 2, 2)
plt.plot(history_3d["val_epoch"], history_3d["val_dice"], marker="o")
plt.xlabel("Epoch"); plt.ylabel("Validation Dice (per volume)")
plt.tight_layout()
plt.show()

In [ ]:
results_3d = evaluate_3d(model_3d, val_loader_3d)
for name, dice, hd in zip(CLASS_NAMES, results_3d["dice"], results_3d["hd95"]):
    print(f"3D U-Net, {name} hippocampus: Dice {dice:.4f}, HD95 {hd:.2f} mm")

### Comparing with the 2D model

To compare the 3D model with the 2D model from the first part of this notebook in a fair way, we also evaluate the 2D model per volume: we put its predictions for the 32 slices of each volume back together and compute the Dice score of each 3D segmentation. (The 2D model works on the resized 32x64x32 volumes, so we only compare the Dice score, which does not depend on the voxel size.)

In [ ]:
def evaluate_2d_per_volume(model, loader):
    """Dice per volume and per class for a 2D model, by stacking its slice predictions back into volumes."""
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
    model.eval()
    with torch.no_grad():
        for data in loader:
            images, labels = data["image"].to(device), data["label"].to(device)
            preds = model(images).argmax(dim=1)  # (volumes * 32 slices, H, W)
            n_volumes = preds.shape[0] // 32
            # custom_collate puts the 32 slices of each volume after each other
            preds = preds.reshape(n_volumes, 32, *preds.shape[1:]).permute(0, 2, 3, 1)
            labels = labels.reshape(n_volumes, 32, *labels.shape[2:]).permute(0, 2, 3, 1).long()
            pred = F.one_hot(preds, num_classes=3).permute(0, 4, 1, 2, 3)
            target = F.one_hot(labels, num_classes=3).permute(0, 4, 1, 2, 3)
            dice_metric(pred, target)
    dice = dice_metric.aggregate().cpu()
    return {"dice": dice.tolist(), "mean_dice": dice.mean().item()}

results_2d = evaluate_2d_per_volume(trained_model, val_loader)
for name, dice in zip(CLASS_NAMES, results_2d["dice"]):
    print(f"2D U-Net (cross-entropy), {name} hippocampus: Dice per volume {dice:.4f}")

## Questions

1. Compare the Dice scores of the 2D and the 3D model per volume. Which one is better, and why do you think that is? Is it a completely fair comparison?
2. Compare the per-volume Dice of the 2D model with the Dice we computed earlier over batches of slices. Why are they different, and which one would you report in a paper?
3. Train the 3D model with and without your data augmentations. How do they change the training loss and the validation Dice?
4. Why do we need sliding window inference at validation time? What would happen if we trained on whole volumes instead of patches for a large CT scan of 512x512x300 voxels?

## Optional part

MONAI also provides implementations of many networks apart from UNET. You can find them in the `monai.networks` module, see [https://docs.monai.io/en/stable/networks.html](https://docs.monai.io/en/stable/networks.html). Try to use one of them for the Hippocampus dataset and evaluate the performance, for example `SegResNet` or the transformer-based `SwinUNETR` in 3D (with `train_3d` and `evaluate_3d`). Compare the results with the U-Net.

For new segmentation problems, self-configuring pipelines such as [nnU-Net](https://github.com/MIC-DKFZ/nnUNet) or MONAI's [Auto3DSeg](https://github.com/Project-MONAI/tutorials/tree/main/auto3dseg) are strong baselines that choose the preprocessing, architecture and training schedule automatically.

In [ ]:
# Your code here